# modular-diffusion: the base run

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/modular-diffusion/blob/v0.1.0/colab/base.ipynb)

Three cells: install the package at a release tag (the runtime keeps its own torch), a 50-step smoke run on Gaussian fields (no data file; its numbers mean nothing), and one run of the base build on CIFAR-10 for 6,000 steps. Use a GPU runtime.

In [ ]:
# C1 - install at a tag; torch is not a dependency, so the runtime's torch stays
%pip install -q "modular-diffusion[data] @ git+https://github.com/AbstractEyes/modular-diffusion@v0.1.0"
import torch, geolip.modular_diffusion as MD
print("geolip.modular_diffusion", MD.__version__, "| torch", torch.__version__, "| card",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert tuple(int(x) for x in MD.__version__.split(".")) >= (0, 1, 0), "RESTART REQUIRED: Runtime > Restart, then rerun C1"
OUT = "/content/modular_diffusion"

In [ ]:
# C2 - a 50-step smoke run on Gaussian fields (no data file); its numbers mean nothing
from geolip.modular_diffusion import train
SMOKE = train.main(bed="fields", steps=50, batch=32, heldout=64, log_every=10, device="cuda", out=f"{OUT}/smoke")

In [ ]:
# C3 - one run: the base build, 6,000 steps
# CIFAR-10 as parquet files from the Hub (one train-*.parquet and one test-*.parquet), then one npz of uint8 arrays
from huggingface_hub import snapshot_download
from geolip.modular_diffusion import data
snapshot_download(repo_id="uoft-cs/cifar10", repo_type="dataset", allow_patterns=["plain_text/*.parquet"], local_dir=f"{OUT}/parquet")
NPZ = data.cifar_npz(parquet_dir=f"{OUT}/parquet/plain_text", out=f"{OUT}/data/cifar10_32.npz")
RUN = train.main(bed="pictures", data=NPZ, bands=3, keep=1, codes="rung", anchor=1, anchor_in=1,
                 seeds="0", steps=6000, device="cuda", out=f"{OUT}/base")